# 1. LangChain Architecture, Chains, Agents & Tools on Amazon Bedrock

## Overview

This notebook introduces the **LangChain architecture** and the three building blocks every
agentic application is made of: **chains**, **tools**, and **agents** — all running on
**Amazon Bedrock**.

The worked example throughout this lab is a **health plan member services assistant**: the kind
of internal tool a payer, a hospital billing office, or a benefits administrator would build to
answer questions about coverage, enrolment and claims.

By the end of this notebook you will be able to:

- Explain how the LangChain package family is layered and why that layering matters
- Connect LangChain to Amazon Bedrock using `ChatBedrockConverse`
- Compose **chains** with LangChain Expression Language (LCEL)
- Define **tools** and bind them to a model
- Build an **agent** that decides on its own which tool to call

> ### Scope of this lab — read this before you start
>
> Everything you build here is **healthcare administration**: benefits, coverage rules,
> enrolment, premiums, claims and appeals. Nothing in this lab is clinical.
>
> The assistant is deliberately built to **refuse clinical questions** — symptoms, diagnosis,
> treatment choice, medication dosing — and redirect them to a licensed professional. That is
> not a disclaimer bolted onto the end; it is a scope guardrail you will implement in the system
> prompts, test in notebook 3, and enforce with a router in notebook 4. Handling out-of-scope
> questions safely is a core engineering skill in this domain, so we treat it as one.
>
> All premium figures, coverage tables and claim records in these notebooks are **illustrative
> sample data for training purposes**. They are not current plan terms, and no output from this
> lab should be used to make a real coverage, billing or care decision.

## Concepts

| Concept | What it is |
|---|---|
| `langchain-core` | The base abstractions — `Runnable`, `Document`, messages, prompts, output parsers |
| `langchain` | Cognitive architecture — agents, chains, retrieval strategies |
| `langchain-aws` | The AWS integration package — `ChatBedrockConverse`, `BedrockEmbeddings` |
| `langchain-community` | Third-party loaders, vector stores and tools |
| **Runnable** | The universal interface. Everything supports `invoke`, `batch`, `stream` |
| **LCEL** | The pipe (`|`) syntax used to compose Runnables into chains |
| **Tool** | A Python function the model is allowed to call |
| **Agent** | A loop: model decides → calls tool → observes result → repeats until done |


## Setup

This notebook runs in **Amazon SageMaker Studio**. Before you start, make sure you have
completed these steps from the lab guide:

1. Requested **model access** in the Amazon Bedrock console, in the same region as your
   SageMaker domain.
2. Attached **`AmazonBedrockFullAccess`** to your **SageMaker execution role**. There are no
   access keys in this lab — the notebook borrows credentials from the role.
3. Created the lab environment and kernel from the JupyterLab terminal:

```bash
cd ~/agentic-ai-healthcare-lab
bash setup_sagemaker.sh
```

4. Selected the **`Python (agentic-lab)`** kernel using the kernel button in the top-right
   corner of this notebook.

> There is no `.env` file to fill in. If cell 1.1 prints your execution role and cell 1.2
> lists models, your configuration is correct.

---
## 1. LangChain Architecture

LangChain is **not one library** — it is a layered family of packages. Understanding the
layers is what makes the rest of the lab easy to reason about.

```
                 +-------------------------------------------+
                 |  Your application (this notebook)          |
                 +-------------------------------------------+
                 |  langchain        chains, agents, retrieval|
                 +-------------------------------------------+
                 |  langchain-aws    langchain-community ...  |   <- integrations
                 +-------------------------------------------+
                 |  langchain-core   Runnable, messages,      |
                 |                   prompts, output parsers  |
                 +-------------------------------------------+
                 |  Amazon Bedrock (Claude, Titan, Nova ...)  |
                 +-------------------------------------------+
```

The single most important abstraction is the **Runnable**. A prompt template is a Runnable.
A chat model is a Runnable. An output parser is a Runnable. A retriever is a Runnable.
Because they all share one interface, they compose with the pipe operator.

#### 1.1. Connect to Bedrock using the SageMaker execution role

In [4]:
# ---------------------------------------------------------------------------
# SageMaker Studio setup
#
# There is no .env file and no access key in this lab. boto3 automatically
# picks up short-lived credentials from the SageMaker execution role attached
# to this space, which is both simpler and safer than a long-lived key pair.
# ---------------------------------------------------------------------------
import os
import boto3

# Leave this as None to use the region this SageMaker space is running in.
# Set it ONLY if you enabled Bedrock model access in a different region.
BEDROCK_REGION = None

AWS_REGION = BEDROCK_REGION or boto3.session.Session().region_name or "us-east-1"
CHAT_MODEL_ID = os.environ.get("BEDROCK_CHAT_MODEL_ID", "amazon.nova-pro-v1:0")
EMBED_MODEL_ID = os.environ.get("BEDROCK_EMBEDDING_MODEL_ID", "amazon.titan-embed-text-v2:0")

session = boto3.Session(region_name=AWS_REGION)

# Show which IAM identity this notebook is using. On SageMaker this is the
# execution role of your space - the role that needs Bedrock permissions.
identity = session.client("sts").get_caller_identity()
EXECUTION_ROLE_ARN = identity["Arn"]

print("Region         :", AWS_REGION)
print("Account        :", identity["Account"])
print("Execution role :", EXECUTION_ROLE_ARN)
print("Chat model     :", CHAT_MODEL_ID)
print("Embed model    :", EMBED_MODEL_ID)

Region         : us-east-1
Account        : 442042521384
Execution role : arn:aws:sts::442042521384:assumed-role/AmazonSageMaker-ExecutionRole-20260807T102279/SageMaker
Chat model     : amazon.nova-pro-v1:0
Embed model    : amazon.titan-embed-text-v2:0


#### 1.2. Verify the execution role can call Bedrock

This is the single most useful cell in the lab. It proves two separate things:

- that your **SageMaker execution role** is allowed to call Bedrock, and
- which **model IDs** your account can actually invoke in this region.

Model IDs change over time and differ by region, so never hard-code one from a blog post.
If `CHAT_MODEL_ID` above does not appear in the output, copy a valid ID from the list into
the setup cell and re-run it.

In [2]:
from botocore.exceptions import ClientError

bedrock = session.client("bedrock")

try:
    fm = bedrock.list_foundation_models(byOutputModality="TEXT")
except ClientError as err:
    print("Bedrock call failed:", err.response["Error"]["Code"])
    print()
    print("The SageMaker execution role below is missing Bedrock permissions:")
    print("   ", EXECUTION_ROLE_ARN)
    print()
    print("Fix it in the IAM console: Roles -> search the role name from that ARN ->")
    print("Add permissions -> Attach policies -> AmazonBedrockFullAccess.")
    raise

print("Execution role can reach Bedrock in", AWS_REGION)

print("\n--- Sample of text models in", AWS_REGION, "---")
for m in fm["modelSummaries"][:15]:
    print(" ", m["modelId"])

# Newer Claude models must be called through a cross-region inference profile
print("\n--- Inference profiles (use these IDs for Claude 4.x) ---")
try:
    profiles = bedrock.list_inference_profiles()
    for p in profiles["inferenceProfileSummaries"][:15]:
        print(" ", p["inferenceProfileId"])
except ClientError as err:
    print("Could not list inference profiles:", err)

Execution role can reach Bedrock in us-east-1

--- Sample of text models in us-east-1 ---
  nvidia.nemotron-nano-12b-v2
  qwen.qwen3-coder-next
  anthropic.claude-sonnet-4-20250514-v1:0
  anthropic.claude-haiku-4-5-20251001-v1:0
  moonshotai.kimi-k2.5
  openai.gpt-oss-120b-1:0
  qwen.qwen3-next-80b-a3b
  anthropic.claude-fable-5
  deepseek.v3.2
  nvidia.nemotron-nano-3-30b
  anthropic.claude-sonnet-4-6
  minimax.minimax-m2
  zai.glm-4.7-flash
  mistral.voxtral-mini-3b-2507
  amazon.nova-pro-v1:0

--- Inference profiles (use these IDs for Claude 4.x) ---


  us.anthropic.claude-3-sonnet-20240229-v1:0
  us.anthropic.claude-3-haiku-20240307-v1:0
  us.meta.llama3-2-11b-instruct-v1:0
  us.meta.llama3-2-3b-instruct-v1:0
  us.meta.llama3-2-90b-instruct-v1:0
  us.meta.llama3-2-1b-instruct-v1:0
  us.meta.llama3-1-8b-instruct-v1:0
  us.meta.llama3-1-70b-instruct-v1:0
  us.meta.llama3-3-70b-instruct-v1:0
  us.deepseek.r1-v1:0
  us.mistral.pixtral-large-2502-v1:0
  us.amazon.nova-premier-v1:0
  us.twelvelabs.pegasus-1-2-v1:0
  us.stability.stable-conservative-upscale-v1:0
  us.twelvelabs.marengo-embed-2-7-v1:0


#### 1.3. Initialize the chat model

`ChatBedrockConverse` uses the Bedrock **Converse API**, which gives one consistent request
shape across every model family on Bedrock (Anthropic, Amazon Nova, Meta, Mistral). It also
supports tool calling, which we need later in this notebook.

In [5]:
from langchain_aws import ChatBedrockConverse

model = ChatBedrockConverse(
    model=CHAT_MODEL_ID,
    client=session.client("bedrock-runtime"),
    temperature=0,
    max_tokens=1024,
)

response = model.invoke(
    "In two sentences, what is the difference between Medicare Part A and Part B?"
)
print(response.content)

Medicare Part A covers hospital insurance, including inpatient care in hospitals, skilled nursing facility care, hospice care, and some home health care. Medicare Part B covers medical insurance for outpatient services, doctor visits, preventive services, and durable medical equipment.


#### 1.4. Messages

Chat models do not take plain strings internally — they take a **list of messages**. LangChain
accepts a string for convenience and converts it. Being explicit gives you control over the
system instruction.

This is also where the **scope guardrail** starts. Notice the second sentence of the system
message: it tells the model what it is *not* for. Write that line first, before you write
anything about what the assistant should do.

In [6]:
from langchain_core.messages import SystemMessage, HumanMessage

BENEFITS_SYSTEM_PROMPT = (
    "You are a health plan member services assistant. You explain benefits, coverage rules, "
    "enrolment, premiums, claims and appeals in plain language. "
    "You do NOT give clinical advice: if asked about symptoms, diagnosis, treatment choice or "
    "medication dosing, say that you cannot help with clinical questions and suggest the member "
    "speak to their doctor or call the nurse line. "
    "Answer in short bullet points."
)

messages = [
    SystemMessage(BENEFITS_SYSTEM_PROMPT),
    HumanMessage("What are the four parts of Medicare, in one line each?"),
]

response = model.invoke(messages)
print(response.content)
print("\nTokens used:", response.usage_metadata)

- **Part A:** Hospital insurance, covers inpatient hospital stays, care in a skilled nursing facility, hospice care, and some home health care.
- **Part B:** Medical insurance, covers certain doctors' services, outpatient care, medical supplies, and preventive services.
- **Part C:** Medicare Advantage, an alternative to Original Medicare offered by private companies, includes Part A and Part B benefits, and often additional benefits.
- **Part D:** Prescription drug coverage, helps cover the cost of prescription drugs, may help lower costs and protect against higher costs.

Tokens used: {'input_tokens': 88, 'output_tokens': 112, 'total_tokens': 200, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}}


In [7]:
# Now the same assistant, given a question that is out of scope.
# Watch it decline and redirect rather than answer.
clinical = [
    SystemMessage(BENEFITS_SYSTEM_PROMPT),
    HumanMessage("I have chest pain and shortness of breath. What should I take for it?"),
]

print(model.invoke(clinical).content)

- I cannot provide clinical advice or recommend medications.
- Please speak to your doctor or call the nurse line for medical advice.
- If you need help understanding your health plan benefits or coverage, I can assist with that.


That refusal is the behaviour we want, and we got it from one sentence in the system prompt.
Two things worth noticing before moving on:

- The refusal is **not reliable enough on its own**. A system prompt is guidance, not
  enforcement. In notebook 3 you will add a retrieval guardrail and in notebook 4 an explicit
  routing node, so the scope check does not depend on the model being in a good mood.
- **Where the line sits is a design decision.** "Is a wheelchair covered under Part B?" is an
  administrative question about a clinical item, and it belongs in scope. "Do I need a
  wheelchair?" does not. Say that out loud in the prompt — the model cannot guess your policy.

#### 1.5. The Runnable interface: invoke, batch, stream

Every LangChain component exposes the same three methods. This is what makes chains
composable — and it is why the *same* code works whether you are calling one model or a
50-step pipeline.

In [8]:
# invoke  -> single input, single output
print("INVOKE:", model.invoke("What does the acronym EOB mean in health insurance?").content)

# batch   -> many inputs in parallel
answers = model.batch([
    "In one sentence: what is a deductible?",
    "In one sentence: what is a copayment?",
])
print("\nBATCH:")
for a in answers:
    print(" -", a.content.strip())

# stream  -> token-by-token
print("\nSTREAM: ", end="")
for chunk in model.stream("Explain prior authorisation in one sentence."):
    print(chunk.text() if hasattr(chunk, "text") else chunk.content, end="")

INVOKE: In health insurance, the acronym EOB stands for "Explanation of Benefits." An EOB is a document provided by health insurance companies to policyholders after they have received medical services. The EOB outlines the details of the claim, including:

1. **Services Received**: A list of the medical services or procedures that were provided.
2. **Charges**: The total amount charged by the healthcare provider for the services.
3. **Allowed Amount**: The amount the insurance company has determined is appropriate to pay for the services, based on the terms of the policy.
4. **Copayment or Coinsurance**: The amount the policyholder is responsible for paying, which could be a copayment (a fixed amount) or coinsurance (a percentage of the allowed amount).
5. **Deductible**: Any portion of the costs that the policyholder must pay out-of-pocket before the insurance coverage kicks in.
6. **Insurance Payment**: The amount the insurance company paid to the healthcare provider.
7. **Patient R


BATCH:
 - A deductible is the amount an insured party must pay out-of-pocket before their insurance coverage begins to pay for covered expenses.
 - A copayment is a fixed amount paid by an insured individual for covered healthcare services at the time of service, in addition to what the insurance company pays.

STREAM: 

Prior authorisation is a process where a patient or healthcare provider must obtain approval from an insurance company before proceeding with certain medical treatments or procedures to ensure coverage.

/home/sagemaker-user/lanchain-healthcare-la/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3775: LangChainDeprecationWarning: Calling .text() as a method is deprecated. Use .text as a property instead (e.g., message.text).
  exec(code_obj, self.user_global_ns, self.user_ns)


---
## 2. Chains in LangChain

A **chain** is a fixed, predictable sequence of steps. You decide the order — the model does
not. Use a chain when you already know exactly what has to happen.

The standard chain shape is:

```
PromptTemplate  ->  ChatModel  ->  OutputParser
```

#### 2.1. Prompt templates

In [9]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You write member-facing benefits explanations for a health plan. "
     "Plain language, no jargon, no clinical advice."),
    ("human", "Write a {length}-word explanation of {topic} for a {audience} audience."),
])

# See what actually gets sent to the model
formatted = prompt.invoke({
    "length": "40",
    "topic": "the Medicare Part B late enrolment penalty",
    "audience": "newly eligible member",
})
for m in formatted.messages:
    print(f"[{m.type}] {m.content}")

[system] You write member-facing benefits explanations for a health plan. Plain language, no jargon, no clinical advice.
[human] Write a 40-word explanation of the Medicare Part B late enrolment penalty for a newly eligible member audience.


#### 2.2. Your first LCEL chain

The pipe operator wires Runnables together. The output of each step becomes the input of the
next.

In [10]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | model | StrOutputParser()

result = chain.invoke({
    "length": "40",
    "topic": "the Medicare Part B late enrolment penalty",
    "audience": "newly eligible member",
})
print(result)

If you don't sign up for Medicare Part B when you're first eligible, you may pay a higher premium later. This is called a late enrolment penalty. It's important to understand this penalty to make informed decisions about your healthcare coverage.


Note that `chain` is itself a Runnable — so it also supports `batch` and `stream`
without any extra code.

In [11]:
results = chain.batch([
    {"length": "30", "topic": "the annual deductible", "audience": "new member"},
    {"length": "30", "topic": "an explanation of benefits", "audience": "new member"},
    {"length": "30", "topic": "the appeals process", "audience": "new member"},
])
for r in results:
    print("-", r.strip(), "\n")

- Your annual deductible is the amount you pay for covered services before your plan starts sharing costs. 

- Your explanation of benefits details what services you used, what you paid, and what your health plan covered, helping you understand your healthcare costs and coverage. 

- If you disagree with a coverage decision, you can appeal. Follow the steps in your denial letter to request a review. 



#### 2.3. Structured output

Free text is hard to use downstream. `with_structured_output()` forces the model to return
data matching a schema, using Bedrock's tool-calling support under the hood.

This is the shape you want when the answer feeds a claims system, a portal widget or a
work-queue — anything that is not a human reading prose.

In [12]:
from pydantic import BaseModel, Field
from typing import Literal


class CoverageAssessment(BaseModel):
    """A structured coverage assessment for a single service or item."""
    service: str = Field(description="The service or item being assessed")
    covered_under: Literal["part_a", "part_b", "part_c", "part_d", "not_covered"] = Field(
        description="Which part of Medicare generally covers this"
    )
    typical_setting: str = Field(description="Where this is usually delivered, one short phrase")
    prior_authorisation_likely: bool = Field(
        description="Whether prior authorisation is commonly required"
    )
    member_action: str = Field(description="One short sentence on what the member should do next")


structured_model = model.with_structured_output(CoverageAssessment)

assessment = structured_model.invoke(
    "Assess coverage for a standard manual wheelchair prescribed for home use."
)
print(type(assessment))
print(assessment)
print("\nCovered under only:", assessment.covered_under)

<class '__main__.CoverageAssessment'>
service='standard manual wheelchair' covered_under='part_b' typical_setting='home use' prior_authorisation_likely=True member_action='Consult with a healthcare provider to determine if prior authorization is needed and to submit a claim.'

Covered under only: part_b


> **A note on trusting this output.** The schema guarantees the *shape*, not the *truth*. The
> model filled `covered_under` from its training data, which may be outdated or simply wrong for
> a specific plan. In notebook 3 we ground exactly this kind of answer in a retrieved source
> document with a page citation. Structure first, grounding second — you need both.

#### 2.4. Multi-step chains: sequencing and parallelism

Real chains rarely have one step. `RunnableParallel` runs branches concurrently and
`RunnableLambda` lets you drop plain Python into the pipeline.

In [13]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough

summarise = ChatPromptTemplate.from_template(
    "Summarise this benefits notice in one sentence a member would understand:\n\n{text}"
) | model | StrOutputParser()

keywords = ChatPromptTemplate.from_template(
    "Extract exactly 5 comma-separated keywords from this text. Return only the keywords:\n\n{text}"
) | model | StrOutputParser()

# Both branches run at the same time against the same input
analyse = RunnableParallel(summary=summarise, keywords=keywords, original=RunnablePassthrough())

sample = (
    "Your plan requires prior authorisation for outpatient advanced imaging, including MRI and "
    "CT scans. Your provider must submit the request before the service is performed. If the "
    "service is performed without an approved authorisation on file, the claim may be denied "
    "and you may be responsible for the full billed amount."
)

out = analyse.invoke({"text": sample})
print("SUMMARY :", out["summary"].strip())
print("KEYWORDS:", out["keywords"].strip())

SUMMARY : Before getting outpatient advanced imaging like MRI or CT scans, your provider must get approval from your plan, or you may have to pay the full cost.
KEYWORDS: authorisation, outpatient, advanced imaging, MRI, CT scans


In [14]:
# RunnableLambda: insert ordinary Python between LLM steps.
# Here it is a crude reading-level check - the kind of automated gate a
# member communications team would put in front of anything they publish.
def flag_long_sentences(text: str) -> str:
    sentences = [s.strip() for s in text.split(".") if s.strip()]
    longest = max((len(s.split()) for s in sentences), default=0)
    verdict = "OK" if longest <= 25 else f"TOO LONG ({longest} words in one sentence)"
    return f"{text}\n\n[readability check: {verdict}]"


checked_chain = summarise | RunnableLambda(flag_long_sentences)
print(checked_chain.invoke({"text": sample}))

Before getting outpatient advanced imaging like MRI or CT scans, your provider must get approval from your plan, or you may have to pay the full cost.

[readability check: TOO LONG (27 words in one sentence)]


---
## 3. Tools

A **tool** is a Python function the model is permitted to call. The `@tool` decorator turns a
function into a schema the model can read: the name, the docstring, and the typed arguments.

Two rules that decide whether your agent works or not:

1. The **docstring is the instruction manual** — the model reads it to decide when to call the tool.
2. **Type hints define the arguments** — the model fills them in, so they must be accurate.

In a benefits assistant, tools are what stop the model from inventing numbers. A premium
penalty is arithmetic with a rule attached; it should never be estimated by a language model
when a function can compute it exactly.

#### 3.1. Define tools

In [15]:
from langchain_core.tools import tool


@tool
def calculate_part_b_late_penalty(standard_premium: float, months_delayed: int) -> str:
    """Calculate the Medicare Part B late enrolment penalty and the resulting monthly premium.

    Use this whenever a member asks what their Part B premium will be after enrolling late,
    or how much the late enrolment penalty costs them.

    standard_premium is the standard monthly Part B premium in USD.
    months_delayed is the total number of months between the end of the member's Initial
    Enrolment Period and the date their Part B coverage starts.

    The penalty is 10 percent of the standard premium for each FULL 12-month period the member
    could have had Part B but did not. Partial years do not count.
    """
    full_years = months_delayed // 12
    penalty_pct = full_years * 10
    penalty_amount = round(standard_premium * penalty_pct / 100, 2)
    total = round(standard_premium + penalty_amount, 2)
    return (
        f"{months_delayed} months delayed = {full_years} full 12-month period(s) "
        f"-> {penalty_pct}% penalty. "
        f"Penalty ${penalty_amount}/month on top of the ${standard_premium} standard premium "
        f"= ${total}/month. This penalty normally continues for as long as the member has Part B."
    )


@tool
def check_service_coverage(service: str) -> str:
    """Look up which part of Medicare generally covers a given service or item.

    Use this when a member asks whether something is covered, or which part pays for it.
    Valid services include: inpatient_hospital, skilled_nursing, doctor_visit,
    outpatient_surgery, durable_medical_equipment, prescription_drugs, routine_dental,
    routine_vision, hearing_aids, annual_wellness_visit.

    This returns sample training data, not live plan terms.
    """
    table = {
        "inpatient_hospital": "Part A - inpatient stay, subject to the Part A deductible per benefit period",
        "skilled_nursing": "Part A - limited days per benefit period, requires a qualifying hospital stay",
        "doctor_visit": "Part B - after the annual deductible, member typically pays 20% coinsurance",
        "outpatient_surgery": "Part B - after the annual deductible, member typically pays 20% coinsurance",
        "durable_medical_equipment": "Part B - 20% coinsurance, supplier must be Medicare-enrolled",
        "prescription_drugs": "Part D - a separate drug plan, or bundled inside a Part C plan",
        "routine_dental": "Not covered by Original Medicare - may be covered by a Part C plan",
        "routine_vision": "Not covered by Original Medicare - may be covered by a Part C plan",
        "hearing_aids": "Not covered by Original Medicare - may be covered by a Part C plan",
        "annual_wellness_visit": "Part B - no cost to the member when the provider accepts assignment",
    }
    key = service.lower().strip().replace(" ", "_")
    return table.get(key, f"No coverage data for '{service}'. Escalate to a benefits specialist.")


print(calculate_part_b_late_penalty.name)
print(calculate_part_b_late_penalty.description)
print(calculate_part_b_late_penalty.args)

calculate_part_b_late_penalty
Calculate the Medicare Part B late enrolment penalty and the resulting monthly premium.

    Use this whenever a member asks what their Part B premium will be after enrolling late,
    or how much the late enrolment penalty costs them.

    standard_premium is the standard monthly Part B premium in USD.
    months_delayed is the total number of months between the end of the member's Initial
    Enrolment Period and the date their Part B coverage starts.

    The penalty is 10 percent of the standard premium for each FULL 12-month period the member
    could have had Part B but did not. Partial years do not count.
{'standard_premium': {'title': 'Standard Premium', 'type': 'number'}, 'months_delayed': {'title': 'Months Delayed', 'type': 'integer'}}


> **Why the docstring is this long.** The penalty rule contains a trap: partial years do not
> count, so 27 months is two full periods, not 2.25. If that rule lives only in the Python and
> not in the docstring, the model will call the tool with the wrong argument or, worse, decide it
> can do the arithmetic itself. Domain rules belong in the description the model reads.

#### 3.2. Binding tools to the model

`bind_tools()` tells the model what is available. Important: the model **does not run** the
tool — it returns a *request* to run it. Look at the `tool_calls` attribute below.

In [16]:
tools = [calculate_part_b_late_penalty, check_service_coverage]
model_with_tools = model.bind_tools(tools)

ai_msg = model_with_tools.invoke(
    "A member delayed Part B enrolment by 27 months. The standard premium is $202.90. "
    "What will they pay each month?"
)

print("Text content :", ai_msg.content)
print("Tool calls   :", ai_msg.tool_calls)

Text content : [{'type': 'reasoning_content', 'reasoning_content': {'text': 'To determine the monthly premium after a late enrolment, I need to calculate the late enrolment penalty first. The penalty is 10 percent of the standard premium for each full 12-month period the member could have had Part B but did not. Since the member delayed enrolment by 27 months, this is equivalent to 2 full 12-month periods (24 months) plus 3 months, which do not count towards the penalty. Therefore, the penalty will be calculated for 2 full periods. I will use the `calculate_part_b_late_penalty` tool to get the penalty and the resulting monthly premium.'}}, {'type': 'tool_use', 'name': 'calculate_part_b_late_penalty', 'input': {'months_delayed': 27, 'standard_premium': 202.9}, 'id': 'tooluse_mhQOOt0OHPyR1ZQQYMBxQG'}]
Tool calls   : [{'name': 'calculate_part_b_late_penalty', 'args': {'months_delayed': 27, 'standard_premium': 202.9}, 'id': 'tooluse_mhQOOt0OHPyR1ZQQYMBxQG', 'type': 'tool_call'}]


#### 3.3. Closing the loop manually

This is exactly what an agent does for you. Doing it once by hand makes the agent loop
completely transparent.

In [17]:
from langchain_core.messages import ToolMessage

conversation = [HumanMessage(
    "A member delayed Part B enrolment by 27 months. The standard premium is $202.90. "
    "What will they pay each month?"
)]

ai_msg = model_with_tools.invoke(conversation)
conversation.append(ai_msg)

tool_registry = {t.name: t for t in tools}

for call in ai_msg.tool_calls:
    selected = tool_registry[call["name"]]
    observation = selected.invoke(call["args"])
    print(f"Ran {call['name']}({call['args']}) -> {observation}")
    conversation.append(ToolMessage(content=str(observation), tool_call_id=call["id"]))

final = model_with_tools.invoke(conversation)
print("\nFINAL ANSWER:", final.content)

Ran calculate_part_b_late_penalty({'months_delayed': 27, 'standard_premium': 202.9}) -> 27 months delayed = 2 full 12-month period(s) -> 20% penalty. Penalty $40.58/month on top of the $202.9 standard premium = $243.48/month. This penalty normally continues for as long as the member has Part B.



FINAL ANSWER: The member will pay $243.48 each month for Medicare Part B. This includes the standard premium of $202.90 plus a late enrolment penalty of $40.58 per month. This penalty will continue for as long as the member has Part B coverage.


---
## 4. Agents

A chain follows the path *you* wrote. An **agent** decides the path itself: it loops between
"call the model" and "run a tool" until the model returns a final answer with no further tool
calls.

Use an agent when the number of steps depends on the question. Member questions are a good fit,
because a single message often contains two or three separate asks.

#### 4.1. Create an agent

In [18]:
from langchain.agents import create_agent

agent = create_agent(
    model,
    tools,
    system_prompt=(
        "You are a health plan member services assistant. "
        "Use the tools available to you rather than estimating numbers or coverage rules. "
        "Always show the figures you used so a specialist can check them. "
        "Do not give clinical advice - redirect clinical questions to the member's doctor. "
        "Remind the member that coverage depends on their specific plan."
    ),
)

result = agent.invoke({"messages": [
    {"role": "user", "content":
     "I put off signing up for Part B for 14 months. If the standard premium is $202.90, "
     "what do I pay?"}
]})

print(result["messages"][-1].content)

Based on the tool result, you delayed signing up for Medicare Part B by 14 months, which is equivalent to 1 full 12-month period. This results in a 10% penalty on the standard premium. 

Here's the breakdown:
- Standard premium: $202.90
- Penalty: 10% of $202.90 = $20.29
- Total monthly premium: $202.90 + $20.29 = $223.19

You will pay $223.19 per month for your Medicare Part B premium. This penalty will continue for as long as you have Part B coverage. Please note that coverage details depend on your specific plan. If you have any clinical questions, please consult your doctor.


#### 4.2. Watch the agent think

Streaming the agent shows every intermediate message: the tool request, the tool result, and
the final answer. This is your primary debugging tool.

The question below deliberately contains **two** asks — a coverage lookup and a premium
calculation. Nowhere did you tell the agent to call two different tools.

In [19]:
query = (
    "Two questions. First, is a wheelchair covered, and under which part? "
    "Second, I enrolled in Part B 30 months late and the standard premium is $202.90 - "
    "what is my monthly cost?"
)

for event in agent.stream({"messages": [{"role": "user", "content": query}]},
                          stream_mode="values"):
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Two questions. First, is a wheelchair covered, and under which part? Second, I enrolled in Part B 30 months late and the standard premium is $202.90 - what is my monthly cost?


================================== Ai Message ==================================

[{'type': 'reasoning_content', 'reasoning_content': {'text': "To answer the user's questions, I need to use the available tools. First, I will check which part of Medicare covers a wheelchair using the `check_service_coverage` tool. Then, I will calculate the user's Part B monthly premium after enrolling 30 months late using the `calculate_part_b_late_penalty` tool."}}, {'type': 'tool_use', 'name': 'check_service_coverage', 'input': {'service': 'durable_medical_equipment'}, 'id': 'tooluse_FB0Ens3iLRzkj3lbqP3hn6'}, {'type': 'tool_use', 'name': 'calculate_part_b_late_penalty', 'input': {'months_delayed': 30, 'standard_premium': 202.9}, 'id': 'tooluse_XBuZ4VVK3bncsSHxL2Zyyo'}]
Tool Calls:
  check_service_coverage (tooluse_FB0Ens3iLRzkj3lbqP3hn6)
 Call ID: tooluse_FB0Ens3iLRzkj3lbqP3hn6
  Args:
    service: durable_medical_equipment
  calculate_part_b_late_penalty (tooluse_XBuZ4VVK3bncsSHxL2Zyyo)
 Call ID: to

================================== Ai Message ==================================

A wheelchair is generally covered under Medicare Part B as durable medical equipment. You will be responsible for a 20% coinsurance, and the supplier must be Medicare-enrolled.

Regarding your Part B enrollment, you enrolled 30 months late. This results in a 20% late enrollment penalty. The standard monthly premium is $202.90, so your penalty is $40.58 per month. Therefore, your total monthly cost for Part B is $243.48. This penalty will continue for as long as you have Part B coverage.

Please note that coverage and costs can vary based on your specific plan, so it's important to review your plan details or contact your plan provider for the most accurate information.


Notice what happened: the agent called **two different tools**, in an order it chose
itself, then combined both results into one answer.

#### 4.3. The agent still respects its scope

Tools do not weaken the guardrail. Run the out-of-scope question through the full agent and
watch it decline without reaching for a tool at all.

In [20]:
out_of_scope = agent.invoke({"messages": [
    {"role": "user", "content":
     "My blood pressure reading was 165/95 this morning. Should I double my medication today?"}
]})

print(out_of_scope["messages"][-1].content)

[{'type': 'reasoning_content', 'reasoning_content': {'text': 'The user is asking for clinical advice, which I am not allowed to provide. I need to redirect them to their doctor for such questions.'}}, {'type': 'text', 'text': '\n\nPlease consult your doctor for any changes to your medication. They can provide the best advice based on your specific health needs and current condition. If you have any questions about your Medicare coverage or need help understanding your plan, feel free to ask.'}]


#### 4.4. Chain or agent? A practical rule

| Use a **chain** when | Use an **agent** when |
|---|---|
| The steps are always the same | The steps depend on the input |
| You want predictable cost and latency | You accept variable cost for flexibility |
| Output must follow a fixed format | The task may need 1 tool or 5 |
| Example: claim notice → plain-language rewrite → readability check | Example: a member message containing three unrelated questions |

Agents are more capable and less predictable. Start with a chain; reach for an agent only when
the branching is genuinely dynamic.

In regulated work there is a third consideration: an agent's path varies between runs, which
makes it harder to evidence what happened during an audit. Notebook 4 shows how LangGraph gets
you dynamic behaviour *and* a replayable trace.

---
## Summary

- LangChain is layered: `langchain-core` → integrations (`langchain-aws`) → `langchain`
- Everything is a **Runnable** with `invoke` / `batch` / `stream`
- **LCEL** composes Runnables with the `|` operator into chains
- **Tools** are functions described to the model by their docstring and type hints
- Domain rules — like "partial years do not count" — belong in the tool description
- **Agents** loop over model + tools and choose their own path
- A scope guardrail starts in the system prompt and gets enforced properly later

**Next:** notebook `2-Memory-and-Context-Management.ipynb`, where we give the assistant a memory
so it can hold a member conversation across turns.